# DistilBERT Multi-Task Model (PyTorch)

Train a single DistilBERT model to predict both Priority and Severity

In [4]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from transformers import (
    DistilBertTokenizer,
    DistilBertModel,
    get_linear_schedule_with_warmup
)

from torch.optim import AdamW  # ✅ FIX HERE

from sklearn.metrics import classification_report, f1_score, accuracy_score
from sklearn.utils.class_weight import compute_class_weight
from tqdm import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
print(f"PyTorch: {torch.__version__}")


Device: cuda
PyTorch: 2.9.0+cu126


## 1. Load Data

In [5]:
train_df = pd.read_csv('/kaggle/input/datasets/ishank9868/github/processed/train.csv')
val_df = pd.read_csv('/kaggle/input/datasets/ishank9868/github/processed/val.csv')
test_df = pd.read_csv('/kaggle/input/datasets/ishank9868/github/processed/test.csv')

print(f"Train: {len(train_df):,}")
print(f"Val:   {len(val_df):,}")
print(f"Test:  {len(test_df):,}")

print("\nPriority distribution:")
print(train_df['priority'].value_counts())
print("\nSeverity distribution:")
print(train_df['severity'].value_counts())

Train: 91,258
Val:   11,407
Test:  11,408

Priority distribution:
priority
low       81549
medium     8106
high       1603
Name: count, dtype: int64

Severity distribution:
severity
Critical    53057
Minor       23969
Major       14232
Name: count, dtype: int64


## 2. Dataset Class

In [6]:
class IssueDataset(Dataset):
    def __init__(self, texts, priority_labels, severity_labels, tokenizer, max_len=256):
        self.texts = texts
        self.priority_labels = priority_labels
        self.severity_labels = severity_labels
        self.tokenizer = tokenizer
        self.max_len = max_len
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        encoding = self.tokenizer(
            text,
            max_length=self.max_len,
            truncation=True,
            padding='max_length',
            return_tensors='pt'
        )
        
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'priority_label': torch.tensor(self.priority_labels[idx], dtype=torch.long),
            'severity_label': torch.tensor(self.severity_labels[idx], dtype=torch.long)
        }

tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

train_dataset = IssueDataset(
    train_df['text'].values,
    train_df['priority_label'].values,
    train_df['severity_label'].values,
    tokenizer
)

val_dataset = IssueDataset(
    val_df['text'].values,
    val_df['priority_label'].values,
    val_df['severity_label'].values,
    tokenizer
)

test_dataset = IssueDataset(
    test_df['text'].values,
    test_df['priority_label'].values,
    test_df['severity_label'].values,
    tokenizer
)

print(" Datasets created")

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

✓ Datasets created


## 3. Multi-Task Model with Improved Architecture

In [7]:
class MultiTaskDistilBERT(nn.Module):
    def __init__(self, n_priority_classes=3, n_severity_classes=3, dropout=0.3):
        super(MultiTaskDistilBERT, self).__init__()
        self.distilbert = DistilBertModel.from_pretrained('distilbert-base-uncased')
        
        hidden_size = self.distilbert.config.hidden_size
        
        # Shared layers
        self.dropout = nn.Dropout(dropout)
        
        # Priority head with additional layer
        self.priority_hidden = nn.Linear(hidden_size, hidden_size // 2)
        self.priority_dropout = nn.Dropout(dropout)
        self.priority_classifier = nn.Linear(hidden_size // 2, n_priority_classes)
        
        # Severity head with additional layer
        self.severity_hidden = nn.Linear(hidden_size, hidden_size // 2)
        self.severity_dropout = nn.Dropout(dropout)
        self.severity_classifier = nn.Linear(hidden_size // 2, n_severity_classes)
        
        self.relu = nn.ReLU()
    
    def forward(self, input_ids, attention_mask):
        outputs = self.distilbert(input_ids=input_ids, attention_mask=attention_mask)
        
        # Use CLS token
        cls_output = outputs.last_hidden_state[:, 0, :]
        cls_output = self.dropout(cls_output)
        
        # Priority branch
        priority_hidden = self.relu(self.priority_hidden(cls_output))
        priority_hidden = self.priority_dropout(priority_hidden)
        priority_logits = self.priority_classifier(priority_hidden)
        
        # Severity branch
        severity_hidden = self.relu(self.severity_hidden(cls_output))
        severity_hidden = self.severity_dropout(severity_hidden)
        severity_logits = self.severity_classifier(severity_hidden)
        
        return priority_logits, severity_logits

model = MultiTaskDistilBERT()
model = model.to(device)
print("Model created with improved architecture")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✓ Model created with improved architecture


## 4. Training Setup with Class Weights

In [8]:
BATCH_SIZE = 16
EPOCHS = 3
LEARNING_RATE = 2e-5

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

# Calculate class weights for imbalanced data
priority_weights = compute_class_weight(
    'balanced',
    classes=np.unique(train_df['priority_label']),
    y=train_df['priority_label']
)
severity_weights = compute_class_weight(
    'balanced',
    classes=np.unique(train_df['severity_label']),
    y=train_df['severity_label']
)

priority_weights = torch.tensor(priority_weights, dtype=torch.float).to(device)
severity_weights = torch.tensor(severity_weights, dtype=torch.float).to(device)

print(f"Priority weights: {priority_weights}")
print(f"Severity weights: {severity_weights}")

# Optimizer with weight decay for regularization
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)

# Learning rate scheduler with warmup
total_steps = len(train_loader) * EPOCHS
warmup_steps = int(0.1 * total_steps)
scheduler = get_linear_schedule_with_warmup(
    optimizer, 
    num_warmup_steps=warmup_steps, 
    num_training_steps=total_steps
)

# Loss functions with class weights
criterion_priority = nn.CrossEntropyLoss(weight=priority_weights)
criterion_severity = nn.CrossEntropyLoss(weight=severity_weights)

print(f"\nTotal training steps: {total_steps}")
print(f"Warmup steps: {warmup_steps}")

Priority weights: tensor([ 0.3730,  3.7527, 18.9765], device='cuda:0')
Severity weights: tensor([1.2691, 2.1374, 0.5733], device='cuda:0')

Total training steps: 17112
Warmup steps: 1711


## 5. Training Loop with Gradient Clipping

In [9]:
def train_epoch(model, data_loader, optimizer, scheduler, device):
    model.train()
    total_loss = 0
    priority_correct = 0
    severity_correct = 0
    total_samples = 0
    
    for batch in tqdm(data_loader, desc="Training"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        priority_labels = batch['priority_label'].to(device)
        severity_labels = batch['severity_label'].to(device)
        
        optimizer.zero_grad()
        
        priority_logits, severity_logits = model(input_ids, attention_mask)
        
        loss_priority = criterion_priority(priority_logits, priority_labels)
        loss_severity = criterion_severity(severity_logits, severity_labels)
        loss = loss_priority + loss_severity
        
        loss.backward()
        
        # Gradient clipping to prevent exploding gradients
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        scheduler.step()
        
        total_loss += loss.item()
        
        # Calculate accuracy
        priority_preds = torch.argmax(priority_logits, dim=1)
        severity_preds = torch.argmax(severity_logits, dim=1)
        priority_correct += (priority_preds == priority_labels).sum().item()
        severity_correct += (severity_preds == severity_labels).sum().item()
        total_samples += priority_labels.size(0)
    
    avg_loss = total_loss / len(data_loader)
    priority_acc = priority_correct / total_samples
    severity_acc = severity_correct / total_samples
    
    return avg_loss, priority_acc, severity_acc

def eval_model(model, data_loader, device):
    model.eval()
    priority_preds = []
    priority_true = []
    severity_preds = []
    severity_true = []
    
    with torch.no_grad():
        for batch in tqdm(data_loader, desc="Evaluating"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            priority_labels = batch['priority_label'].to(device)
            severity_labels = batch['severity_label'].to(device)
            
            priority_logits, severity_logits = model(input_ids, attention_mask)
            
            priority_preds.extend(torch.argmax(priority_logits, dim=1).cpu().numpy())
            priority_true.extend(priority_labels.cpu().numpy())
            severity_preds.extend(torch.argmax(severity_logits, dim=1).cpu().numpy())
            severity_true.extend(severity_labels.cpu().numpy())
    
    priority_f1 = f1_score(priority_true, priority_preds, average='weighted')
    severity_f1 = f1_score(severity_true, severity_preds, average='weighted')
    priority_acc = accuracy_score(priority_true, priority_preds)
    severity_acc = accuracy_score(severity_true, severity_preds)
    
    return priority_f1, severity_f1, priority_acc, severity_acc

print(" Training functions defined")

✓ Training functions defined


## 6. Train Model with Early Stopping

In [10]:
best_val_f1 = 0
patience = 2
patience_counter = 0

for epoch in range(EPOCHS):
    print(f"\n{'='*60}")
    print(f"Epoch {epoch + 1}/{EPOCHS}")
    print(f"{'='*60}")
    
    train_loss, train_priority_acc, train_severity_acc = train_epoch(
        model, train_loader, optimizer, scheduler, device
    )
    
    print(f"\nTrain Loss: {train_loss:.4f}")
    print(f"Train Priority Acc: {train_priority_acc:.4f}")
    print(f"Train Severity Acc: {train_severity_acc:.4f}")
    
    priority_f1, severity_f1, priority_acc, severity_acc = eval_model(model, val_loader, device)
    
    print(f"\nVal Priority F1: {priority_f1:.4f} | Acc: {priority_acc:.4f}")
    print(f"Val Severity F1: {severity_f1:.4f} | Acc: {severity_acc:.4f}")
    
    # Save best model
    avg_f1 = (priority_f1 + severity_f1) / 2
    if avg_f1 > best_val_f1:
        best_val_f1 = avg_f1
        torch.save(model.state_dict(), 'best_model.pth')
        print(f"✓ Best model saved! Avg F1: {avg_f1:.4f}")
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"\nEarly stopping triggered after {epoch + 1} epochs")
            break

print("\n✓ Training complete")


Epoch 1/3


Training: 100%|██████████| 5704/5704 [36:12<00:00,  2.63it/s]



Train Loss: 1.6590
Train Priority Acc: 0.7354
Train Severity Acc: 0.7207


Evaluating: 100%|██████████| 713/713 [01:28<00:00,  8.01it/s]



Val Priority F1: 0.8092 | Acc: 0.7808
Val Severity F1: 0.7804 | Acc: 0.7680
✓ Best model saved! Avg F1: 0.7948

Epoch 2/3


...


Val Priority F1: 0.8128 | Acc: 0.7797
Val Severity F1: 0.7867 | Acc: 0.7697
✓ Best model saved! Avg F1: 0.7997

Epoch 3/3


Training: 100%|██████████| 5704/5704 [36:31<00:00,  2.60it/s]



Train Loss: 1.2972
Train Priority Acc: 0.8011
Train Severity Acc: 0.8005


Evaluating: 100%|██████████| 713/713 [01:29<00:00,  8.01it/s]


Val Priority F1: 0.8083 | Acc: 0.7728
Val Severity F1: 0.7899 | Acc: 0.7739

✓ Training complete


## 7. Load Best Model and Test

In [11]:
# Load best model
model.load_state_dict(torch.load('best_model.pth'))
print("✓ Loaded best model")

priority_f1, severity_f1, priority_acc, severity_acc = eval_model(model, test_loader, device)

print(f"\n{'='*60}")
print("TEST SET RESULTS")
print(f"{'='*60}")
print(f"Priority F1: {priority_f1:.4f} | Accuracy: {priority_acc:.4f}")
print(f"Severity F1: {severity_f1:.4f} | Accuracy: {severity_acc:.4f}")
print(f"Average F1: {(priority_f1 + severity_f1) / 2:.4f}")


✓ Loaded best model


Evaluating: 100%|██████████| 713/713 [01:29<00:00,  8.01it/s]


TEST SET RESULTS
Priority F1: 0.8124 | Accuracy: 0.7782
Severity F1: 0.7893 | Accuracy: 0.7735
Average F1: 0.8008


## 8. Detailed Classification Reports

In [12]:
model.eval()
priority_preds = []
priority_true = []
severity_preds = []
severity_true = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        priority_labels = batch['priority_label'].to(device)
        severity_labels = batch['severity_label'].to(device)
        
        priority_logits, severity_logits = model(input_ids, attention_mask)
        
        priority_preds.extend(torch.argmax(priority_logits, dim=1).cpu().numpy())
        priority_true.extend(priority_labels.cpu().numpy())
        severity_preds.extend(torch.argmax(severity_logits, dim=1).cpu().numpy())
        severity_true.extend(severity_labels.cpu().numpy())

print("\nPRIORITY Classification Report:")
print(classification_report(priority_true, priority_preds, target_names=['low', 'medium', 'high']))

print("\nSEVERITY Classification Report:")
print(classification_report(severity_true, severity_preds, target_names=['Minor', 'Major', 'Critical']))


PRIORITY Classification Report:
              precision    recall  f1-score   support

         low       0.94      0.82      0.88     10194
      medium       0.21      0.48      0.29      1013
        high       0.26      0.26      0.26       201

    accuracy                           0.78     11408
   macro avg       0.47      0.52      0.47     11408
weighted avg       0.86      0.78      0.81     11408


SEVERITY Classification Report:
              precision    recall  f1-score   support

       Minor       0.66      0.69      0.68      2921
       Major       0.43      0.63      0.51      1816
    Critical       0.99      0.85      0.91      6671

    accuracy                           0.77     11408
   macro avg       0.69      0.72      0.70     11408
weighted avg       0.82      0.77      0.79     11408



## 9. Save Final Model

In [13]:
torch.save(model.state_dict(), 'distilbert_multitask_final.pth')
tokenizer.save_pretrained('tokenizer')

print(" Model saved to distilbert_multitask_final.pth")
print(" Tokenizer saved to tokenizer/")


✓ Model saved to distilbert_multitask_final.pth
✓ Tokenizer saved to tokenizer/


## Summary

**Model**: DistilBERT Multi-Task (PyTorch)

**Improvements for Accuracy**:
1. Class weights for imbalanced data
2. Deeper classification heads (2 layers each)
3. Gradient clipping (max_norm=1.0)
4. Weight decay (0.01) for regularization
5. Learning rate warmup (10% of steps)
6. Early stopping with patience=2
7. Best model checkpointing

**Architecture**:
- Shared DistilBERT base (66M params)
- 2 task-specific heads with hidden layers
- Dropout (0.3) for regularization

**Training**:
- Up to 3 epochs with early stopping
- Batch size: 16
- Learning rate: 2e-5

**Expected Performance**:
- Priority F1: 0.88-0.92
- Severity F1: 0.78-0.88